# MedGemma Ollama Playground

Run MedGemma locally via Ollama — no GPU headaches, no CUDA errors.

## 1. Setup & Config

In [1]:
MODEL = "hf.co/unsloth/medgemma-1.5-4b-it-GGUF"
# MODEL = "gemma3"

MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0

In [3]:
%pip install "ollama>=0.4.0" "pydantic>=2.0.0" "gradio>=4.0.0" "Pillow>=10.0.0" "numpy>=1.24.0" "langgraph>=0.2.0" "langchain-core>=0.2.0"


In [13]:
import subprocess
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh


subprocess.Popen("ollama serve", shell=True)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (309 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 121852 files and directories currently i

<Popen: returncode: None args: 'ollama serve'>

In [14]:
%pip install -q ollama langchain-ollama

import ollama

# Check connection & pull model if needed
try:
    models = [m.model for m in ollama.list().models]
    print(f"Ollama is running. Models: {models}")
    if not any(MODEL in m for m in models):
        print(f"Pulling {MODEL}...")
        ollama.pull(MODEL)
        print("Done.")
    else:
        print(f"{MODEL} is ready.")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure Ollama is running: https://ollama.com")

Ollama is running. Models: []
Pulling hf.co/unsloth/medgemma-1.5-4b-it-GGUF...
Done.


## 2. Helpers

In [15]:
import base64
from io import BytesIO
from PIL import Image


def generate(prompt: str, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE) -> str:
    """Send a text prompt and return the response."""
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"num_predict": max_tokens, "temperature": temperature},
    )
    tokens_in = response.prompt_eval_count or 0
    tokens_out = response.eval_count or 0
    print(f"[{tokens_in} input tokens -> {tokens_out} output tokens]")
    return response.message.content


def generate_with_image(image: Image.Image, prompt: str, max_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE) -> str:
    """Send an image + text prompt and return the response."""
    buf = BytesIO()
    image.save(buf, format="PNG")
    img_b64 = base64.b64encode(buf.getvalue()).decode()

    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt, "images": [img_b64]}],
        options={"num_predict": max_tokens, "temperature": temperature},
    )
    tokens_in = response.prompt_eval_count or 0
    tokens_out = response.eval_count or 0
    print(f"[{tokens_in} input tokens -> {tokens_out} output tokens]")
    return response.message.content

In [16]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL, temperature=TEMPERATURE, max_tokens=MAX_NEW_TOKENS)

## 3. Basic Text Generation

In [17]:
response = llm.invoke("What are the symptoms of malaria?")
print(response)

content='<unused94>thought\nHere\'s a thinking process for generating the answer about malaria symptoms:\n\n1.  **Understand the User\'s Request:** The user wants to know the symptoms of malaria. This is a straightforward request asking for information about a specific medical condition.\n\n2.  **Identify Key Information Needed:** To provide a comprehensive and helpful answer, I need to cover:\n    *   What malaria is (briefly).\n    *   The typical onset time of symptoms.\n    *   Common symptoms.\n    *   Less common but serious complications/symptoms.\n    *   Symptoms specific to certain types of malaria (e.g., *P. falciparum* vs. *P. vivax*).\n    *   Important caveats and advice (seek medical help, diagnosis confirmation).\n\n3.  **Structure the Answer:** A logical structure will make the information easier to understand. I\'ll use headings or bullet points:\n    *   Introduction/Brief Definition\n    *   Typical Onset Time\n    *   Common Symptoms (the core of the answer)\n    *

In [18]:
response = ollama.ps()
print(response)

models=[Model(model='hf.co/unsloth/medgemma-1.5-4b-it-GGUF:latest', name='hf.co/unsloth/medgemma-1.5-4b-it-GGUF:latest', digest='658c374318f07b0c6798e6ac485be9079138e70cbb6c36820d8b1fbe7d0b4aa5', expires_at=datetime.datetime(2026, 2, 20, 21, 17, 0, 751913, tzinfo=TzInfo(0)), size=7784957184, size_vram=7784957184, details=ModelDetails(parent_model='', format='gguf', family='gemma3', families=['gemma3'], parameter_size='3.88B', quantization_level='unknown'), context_length=131072)]


## 4. Structured Output (LangChain + Pydantic)

Use `with_structured_output` to get validated Pydantic objects directly from the model.

In [25]:
from pydantic import BaseModel, Field
from typing import List, Optional


class Medication(BaseModel):
    name: str
    dosage: str
    duration: Optional[str] = None


class ClinicalAssessment(BaseModel):
    condition: str = Field(description="Primary clinical condition or diagnosis")
    confidence: str = Field(description="Confidence level of the diagnosis (High, Medium, Low)")
    differential_diagnoses: List[str] = Field(default_factory=list, description="List of alternative diagnoses to consider")
    known_symptoms: List[str] = Field(default_factory=list, description="Symptoms observed in the patient")
    treatment: List[Medication] = Field(default_factory=list, description="Recommended medications and treatment plan")
    instructions: List[str] = Field(default_factory=list, description="Patient care instructions and management steps")
    warning_signs: List[str] = Field(default_factory=list, description="Critical warning signs that require immediate referral")
    follow_up_days: int = Field(default=3, description="Number of days until follow-up appointment")
    requires_referral: bool = Field(default=False, description="Whether patient needs referral to higher facility")
    referral_reason: Optional[str] = Field(default=None, description="Reason for referral if applicable")


structured_llm = llm.with_structured_output(ClinicalAssessment)

In [26]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=ClinicalAssessment)

In [27]:
import json
from langchain_core.prompts import PromptTemplate

symptoms = "Patient has had high fever for 3 days, shaking chills especially at night, severe headache, body aches, and sweating. Lives in malaria-endemic area."
patient_age = "adult"
current_medications = ["Metformin", "Amlodipine"]

template ="""
    You are a medical decision support system for a Community Health Worker.\n
    Analyze the patient case.
    {format_instructions}\n

    PATIENT CASE:\n
    Age: {patient_age}\n
    Symptoms: {symptoms}\n
    Current medications: {current_medications}\n
"""


prompt = PromptTemplate(
    template=template,
    input_variables=["patient_age", "symptoms", "current_medications"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
    )

chain =  prompt | structured_llm
assessment = chain.invoke({
    "patient_age": patient_age,
    "symptoms": symptoms,
    "current_medications": current_medications
})

In [28]:
assessment

ClinicalAssessment(condition='Malaria', confidence='High', differential_diagnoses=['Typhoid fever', 'Dengue fever', 'Viral hepatitis'], known_symptoms=['High fever', 'Shaking chills', 'Severe headache', 'Body aches', 'Sweating'], treatment=[Medication(name='Artemisinin-based combination therapy (ACT)', dosage='1.5 mg/kg body weight per day', duration=None), Medication(name='Hydroxychloroquine', dosage='200 mg orally once daily', duration=None)], instructions=['Take medication as prescribed.', 'Rest in a well-ventilated area.', 'Stay hydrated by drinking plenty of fluids.', 'Seek immediate medical attention if symptoms worsen or fever persists beyond 48 hours.'], warning_signs=['Severe abdominal pain', 'Difficulty breathing', 'Confusion or altered mental state', 'Signs of severe anemia (e.g., rapid heart rate, shortness of breath)'], follow_up_days=3, requires_referral=False, referral_reason=None)

In [29]:
# Load a test image — replace with your own path
# image = Image.open("path/to/medical_image.jpg")

# Or create a dummy image to test the pipeline works
image = Image.new("RGB", (224, 224), color=(200, 150, 150))
print(f"Image size: {image.size}")

Image size: (224, 224)


In [30]:
vision_response = generate_with_image(
    image,
    "Describe any medical findings visible in this image. "
    "List each finding on its own line.",
    max_tokens=256,
)
print(vision_response)

[285 input tokens -> 54 output tokens]
Based on the image provided, there are no discernible medical findings. It appears to be a solid color block with no anatomical structures or abnormalities visible.

If you have a different image or require assistance interpreting medical images, please provide more details or upload the image again.


## 6. Free Experimentation

Try your own prompts below.

In [ ]:
response = generate("What is the first-line treatment for uncomplicated malaria in sub-Saharan Africa?")
print(response)